# EfficientNetV2 + Grad-CAM for Oral Histopathology Classification

## 1. Environment & Dependencies

Input > Add Input >  
https://www.kaggle.com/datasets/sthephannysantos/oral-dataset

Esta seção verifica o ambiente, instala `timm` e `thop`, e confirma a GPU.

In [ ]:
# Montar os dados:
import os
print('Kaggle input:', os.listdir('/kaggle/input'))

In [ ]:
# Instalar dependências extras
import subprocess
subprocess.run(['pip', 'install', 'timm', 'thop', '-q'], check=True)

In [ ]:
# Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Imports
Todas as bibliotecas necessárias.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch import optim
import timm
import os
import copy
import json
import time
import random

import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from thop import profile

print('Todas as importações OK.')

## 3. Project Settings (ALTERADO)

Definições globais:
- Caminhos dos datasets e resultados.
- **LR fixo por dataset** (sem scheduler).
- Early stopping com paciência definida por modo de treinamento.
- Grade de experimentos: 2 arquiteturas × 3 modos × 2 augmentations × 3 seeds = 72.

In [ ]:
import torch, os

# ── Caminhos Kaggle ──────────────────────────────────────────────────────────
BASE_INPUT  = '/kaggle/input/datasets/sthephannysantos/oral-dataset'
DATASET_DIR = f'{BASE_INPUT}/datasets'
RESULTS_DIR = '/kaggle/working'

# ── Configurações gerais ─────────────────────────────────────────────────────
definitions = {
    'datasets_folder': DATASET_DIR + '/',
    'imgs1_folder':    DATASET_DIR + '/Original ROI images/',
    'imgs2_folder':    DATASET_DIR + '/images/',
    'batch_size':      32,
    'num_workers':     2,
}

# ── Learning rates fixos por dataset (sem scheduler) ────────────────────── # ← ALTERADO
lr_per_dataset = {
    'NDB-UFES':         1e-5,    # valor sugerido para câncer oral
    'OralEpitheliumDB': 1e-5,    # valor mais conservador para displasia
}

# ── Hiperparâmetros por modo de treinamento ────────────────────────────────
epochs_map = {
    'scratch':            50,
    'feature_extraction': 50,
    'fine_tuning':        50,
}
es_patience_map = {                                                        
    'scratch':            30,
    'feature_extraction': 30,
    'fine_tuning':        30,
}

# ── Experimentos ─────────────────────────────────────────────────────────────
models        = ['tf_efficientnetv2_b0.in1k', 'tf_efficientnetv2_b1.in1k']
training_modes     = ['scratch', 'feature_extraction', 'fine_tuning']
augmentation_modes = [True, False]
seeds              = [42, 123, 2025]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

# ── Criar pastas de saída ─────────────────────────────────────────────────────
for subdir in ['checkpoints', 'histories', 'confusion_matrices',
               'learning_curves', 'gradcam', 'graficos']:
    os.makedirs(f'{RESULTS_DIR}/{subdir}', exist_ok=True)

print(f'Pasta de resultados: {RESULTS_DIR}')
print(f'Conteúdo do dataset: {os.listdir(DATASET_DIR)}')

## 4. Transforms & Dataset

Pré‑processamento de imagens (augmentation opcional) e classe `OralCancerDataset`.

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomChoice([
        transforms.RandomRotation(degrees=(0,   0)),
        transforms.RandomRotation(degrees=(90,  90)),
        transforms.RandomRotation(degrees=(180, 180)),
        transforms.RandomRotation(degrees=(270, 270)),
    ]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_validation = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def get_transform_train(augmentation=True):
    return transform_train if augmentation else transform_validation

class OralCancerDataset(Dataset):
    def __init__(self, df, images_folder, transform=None):
        self.df            = df
        self.images_folder = images_folder
        self.transform     = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(os.path.join(self.images_folder, row['path'])).convert('RGB')
        label = int(row['label_number'])
        if self.transform:
            img = self.transform(img)
        return img, label

print('Dataset e transformações prontos.')

## 5. Utility Functions

Funções auxiliares: seed, criação de modelo, complexidade, denormalização, pesos de classe.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def create_model(architecture, mode, num_classes=3):
    if mode == 'scratch':
        model = timm.create_model(architecture, pretrained=False, num_classes=num_classes)
    elif mode == 'feature_extraction':
        model = timm.create_model(architecture, pretrained=True, num_classes=num_classes)
        for p in model.parameters():
            p.requires_grad = False
        for p in model.get_classifier().parameters():
            p.requires_grad = True
    elif mode == 'fine_tuning':
        model = timm.create_model(architecture, pretrained=True, num_classes=num_classes)
    return model

def get_model_complexity(model, device):
    num_params = sum(p.numel() for p in model.parameters())
    dummy      = torch.zeros(1, 3, 224, 224).to(device)
    model.eval()
    flops, _   = profile(model, inputs=(dummy,), verbose=False)
    return num_params, flops / 1e9

def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img  = tensor.cpu() * std + mean
    return np.clip(img.permute(1, 2, 0).numpy(), 0, 1)

def get_class_weights(train_loader, device):
    all_labels = []
    for _, labels in train_loader:
        all_labels.extend(labels.cpu().numpy())
    classes, counts = np.unique(all_labels, return_counts=True)
    total_samples = len(all_labels)
    num_classes = len(classes)
    weights = total_samples / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32).to(device)

def save_checkpoint(base_dir, dataset_name, weights, model_name, mode, aug, seed):
    checkpoint_dir = os.path.join(base_dir, 'checkpoints')
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, f"{dataset_name}_{model_name}_{mode}_aug{aug}_seed{seed}.pth")
    torch.save(weights, path)

def save_execution(base_dir, dataset_name, model_name, mode, aug, seed, history, test_metrics):
    name = f"{dataset_name}_{model_name}_{mode}_aug{aug}_seed{seed}"
    hist_dir = os.path.join(base_dir, 'histories')
    cm_dir   = os.path.join(base_dir, 'confusion_matrices')
    os.makedirs(hist_dir, exist_ok=True)
    os.makedirs(cm_dir, exist_ok=True)

    with open(os.path.join(hist_dir, f"{name}.json"), "w") as f:
        json.dump(history, f)

    with open(os.path.join(cm_dir, f"{name}.json"), "w") as f:
        json.dump(test_metrics["confusion_matrix"], f)

print('Utilitários prontos.')

## 6. Training & Evaluation Functions

Funções de treino, validação e teste (com mixed precision).

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        correct    += (outputs.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)
          # Train Loss                Train Accuracy
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            loss           = criterion(outputs, labels)
            total_loss    += loss.item()
            correct       += (outputs.argmax(dim=1) == labels).sum().item()
            total         += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate_test(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds  = model(images).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return {
        'acc_test':         accuracy_score(all_labels, all_preds),
        'f1_macro_test':    f1_score(all_labels, all_preds, average='macro'),
        'f1_weighted_test': f1_score(all_labels, all_preds, average='weighted'),
        'confusion_matrix': confusion_matrix(all_labels, all_preds).tolist(),
        'all_preds':        all_preds,
        'all_labels':       all_labels,
    }

print('Funções de treinamento prontas.')

## 7. Visualization Functions

Gráficos de curvas de aprendizado, matrizes de confusão e resumos.

In [ ]:
def plot_learning_curves(history, name, save_dir):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history['train_loss'], label='Train')
    axes[0].plot(epochs, history['val_loss'],   label='Val')
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Curva de Loss'); axes[0].legend()
    axes[1].plot(epochs, history['train_acc'], label='Train')
    axes[1].plot(epochs, history['val_acc'],   label='Val')
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia')
    axes[1].set_title('Curva de Acurácia'); axes[1].legend()
    fig.suptitle(name.replace('_', ' '), fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/learning_curves/{name}.png', dpi=100, bbox_inches='tight')
    plt.close()

def plot_confusion_matrix(cm_list, num_classes, class_names, name, save_dir):
    cm   = np.array(cm_list)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=class_names[:num_classes])
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name.replace('_', ' '))
    plt.tight_layout()
    plt.savefig(f'{save_dir}/confusion_matrices/{name}.png', dpi=100, bbox_inches='tight')
    plt.close()

def generate_summary_plots(df, save_dir):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    df.boxplot(column='f1_weighted_test', by='model', ax=axes[0])
    axes[0].set_title('F1 Weighted por Arquitetura')
    axes[0].set_xlabel('Arquitetura'); axes[0].set_ylabel('F1 Weighted')
    df.boxplot(column='f1_weighted_test', by='training_mode', ax=axes[1])
    axes[1].set_title('F1 Weighted por Modo')
    axes[1].set_xlabel('Modo'); axes[1].set_ylabel('')
    axes[1].tick_params(axis='x', rotation=15)
    df.boxplot(column='f1_weighted_test', by='augmentation', ax=axes[2])
    axes[2].set_title('F1 Weighted por Augmentation')
    axes[2].set_xlabel('Augmentation'); axes[2].set_ylabel('')
    fig.suptitle('')
    plt.tight_layout()
    plt.savefig(f'{save_dir}/graficos/f1_comparativos.png', dpi=100, bbox_inches='tight')
    plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    arch_stats = df.groupby('model')[['num_params', 'gflops']].first()
    arch_stats['num_params'].plot(kind='bar', ax=axes[0], color='steelblue', rot=15)
    axes[0].set_title('Número de Parâmetros')
    axes[0].set_ylabel('Parâmetros')
    axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    arch_stats['gflops'].plot(kind='bar', ax=axes[1], color='darkorange', rot=15)
    axes[1].set_title('Complexidade Computacional (GFLOPs)')
    axes[1].set_ylabel('GFLOPs')
    plt.tight_layout()
    plt.savefig(f'{save_dir}/graficos/complexidade_modelos.png', dpi=100, bbox_inches='tight')
    plt.show()

print('Funções de visualização prontas.')

## 8. Grad-CAM

Implementação completa de Grad‑CAM para interpretabilidade.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.activations  = None
        self.gradients    = None
        self._fwd = target_layer.register_forward_hook(self._save_activations)
        self._bwd = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(input_tensor)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()
        logits[:, class_idx].sum().backward()
        if self.activations is None or self.gradients is None:
            raise RuntimeError('Ativações/gradientes não capturados.')
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam     = F.interpolate(cam, size=input_tensor.shape[-2:],
                                mode='bilinear', align_corners=False)
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        prob = torch.softmax(logits, dim=1)[0, class_idx].item()
        return cam.squeeze().cpu(), class_idx, prob

    def remove_hooks(self):
        self._fwd.remove()
        self._bwd.remove()

def get_efficientnet_target_layer(model):
    if hasattr(model, 'conv_head'):
        return model.conv_head
    if hasattr(model, 'blocks') and len(model.blocks) > 0:
        last_block = model.blocks[-1]
        if isinstance(last_block, nn.Sequential):
            for m in reversed(list(last_block.children())):
                if isinstance(m, nn.Conv2d):
                    return m
        return last_block
    for m in reversed(list(model.children())):
        if not isinstance(m, (nn.Linear, nn.AdaptiveAvgPool2d, nn.Flatten, nn.Identity)):
            return m
    raise ValueError('Camada alvo não encontrada.')

def select_gradcam_samples(model, dataloader, device, n_per_class=2):
    model.eval()
    images_list, labels_list, preds_list = [], [], []
    with torch.no_grad():
        for images, labels in dataloader:
            batch_preds = model(images.to(device)).argmax(dim=1).cpu()
            for i in range(len(labels)):
                images_list.append(images[i])
                labels_list.append(labels[i].item())
                preds_list.append(batch_preds[i].item())
    classes = sorted(set(labels_list))
    selected = []
    for cls in classes:
        idx_correct   = [i for i in range(len(labels_list)) if labels_list[i] == cls and preds_list[i] == cls]
        idx_incorrect = [i for i in range(len(labels_list)) if labels_list[i] == cls and preds_list[i] != cls]
        rng = np.random.default_rng(42)
        rng.shuffle(idx_correct)
        rng.shuffle(idx_incorrect)
        if len(idx_correct) >= 1 and len(idx_incorrect) >= 1:
            chosen_idx = [idx_correct[0], idx_incorrect[0]]
        elif len(idx_correct) >= n_per_class:
            chosen_idx = idx_correct[:n_per_class]
        elif len(idx_incorrect) >= n_per_class:
            chosen_idx = idx_incorrect[:n_per_class]
        else:
            chosen_idx = (idx_correct + idx_incorrect)[:n_per_class]
        for i in chosen_idx:
            selected.append({
                'image':   images_list[i],
                'label':   labels_list[i],
                'pred':    preds_list[i],
                'correct': labels_list[i] == preds_list[i],
            })
    return selected

def visualize_gradcam(model, target_layer, sample, class_names, idx, save_dir):
    img_tensor      = sample['image'].unsqueeze(0).to(device)
    true_label      = sample['label']
    predicted_label = sample['pred']
    is_correct      = sample['correct']
    img_display = denormalize(sample['image'])

    def _run_gradcam(class_idx):
        gc  = GradCAM(model, target_layer)
        with torch.enable_grad():
            cam, _, prob = gc.generate(img_tensor, class_idx=class_idx)
        gc.remove_hooks()
        return cam, prob

    cam_pred, prob_pred = _run_gradcam(predicted_label)
    n_cols = 3 if not is_correct else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))

    def _plot(ax, cam, title):
        ax.imshow(img_display)
        ax.imshow(cam.numpy(), cmap='jet', alpha=0.45)
        ax.axis('off')
        ax.set_title(title, fontsize=9)

    true_name = class_names[true_label] if class_names else str(true_label)
    pred_name = class_names[predicted_label] if class_names else str(predicted_label)
    axes[0].imshow(img_display)
    axes[0].axis('off')
    status = '✓ Correto' if is_correct else '✗ Incorreto'
    axes[0].set_title(f'Original\nVerdadeiro: {true_name}\n{status}', fontsize=9)
    _plot(axes[1], cam_pred, f'Grad-CAM (Previsto)\n{pred_name} ({prob_pred:.2%})')
    if not is_correct:
        cam_true, prob_true = _run_gradcam(true_label)
        _plot(axes[2], cam_true, f'Grad-CAM (Verdadeiro)\n{true_name} ({prob_true:.2%})')
    plt.tight_layout()
    status_str = 'correct' if is_correct else 'incorrect'
    fname = f'sample{idx:02d}_cls{true_label}_{status_str}_pred{predicted_label}.png'
    plt.savefig(f'{save_dir}/{fname}', dpi=100, bbox_inches='tight')
    plt.show()
    plt.close('all')
    import gc; gc.collect()

print('Grad-CAM pronto.')

## 9. Main Training Loop

- Carrega os manifests e configura os datasets.
- **Loop principal**: para cada combinação (dataset, modelo, modo, augmentation, seed):
    1. Seta a semente.
    2. Cria os DataLoaders (com augmentation opcional).
    3. Instancia o modelo e a função de perda (com pesos para NDB‑UFES).
    4. Usa **LR fixo** (obtido de `lr_per_dataset`) e **Adam** (sem scheduler).
    5. Treina com **early stopping** baseado em `val_acc` (paciente conforme `es_patience_map`).
    6. Avalia no teste, salva checkpoint, histórico, matriz de confusão e curvas.
    7. Registra resultados em CSV incremental.


In [ ]:
# Carregar manifests
df_epithelium = pd.read_csv(definitions['datasets_folder'] + 'manifest_split_oralepitheliumdb.csv')
df_cancer     = pd.read_csv(definitions['datasets_folder'] + 'manifest_split_multiclass_NDB-UFES.csv')

datasets_config = [
    {
        'name':        'OralEpitheliumDB',
        'train':       df_epithelium[df_epithelium['sets'] == 'train'].reset_index(drop=True),
        'val':         df_epithelium[df_epithelium['sets'] == 'val'].reset_index(drop=True),
        'test':        df_epithelium[df_epithelium['sets'] == 'test'].reset_index(drop=True),
        'folder':      definitions['imgs1_folder'],
        'num_classes': 2,
        'class_names': ['Normal', 'Dysplasia'],
    },
    {
        'name':        'NDB-UFES',
        'train':       df_cancer[df_cancer['sets'] == 'train'].reset_index(drop=True),
        'val':         df_cancer[df_cancer['sets'] == 'val'].reset_index(drop=True),
        'test':        df_cancer[df_cancer['sets'] == 'test'].reset_index(drop=True),
        'folder':      definitions['imgs2_folder'],
        'num_classes': 3,
        'class_names': ['Normal', 'Low-grade', 'High-grade'],
    },
]

print('Datasets carregados:')
for ds in datasets_config:
    total = len(ds['train']) + len(ds['val']) + len(ds['test'])
    print(f"  {ds['name']}: {total} imagens "
          f"(train={len(ds['train'])}, val={len(ds['val'])}, test={len(ds['test'])})")

In [ ]:
def training(model, loader_train, loader_val, loader_test, criterion, optimizer,
             epochs, model_name, mode, aug, seed, dataset_name, base_dir, device):
    
    scaler = GradScaler()  # <-- instanciado aqui
    best_val_acc, best_epoch = 0.0, 0
    best_weights = copy.deepcopy(model.state_dict())
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, loader_train, criterion, optimizer, scaler, device)
        val_loss, val_acc     = validate(model, loader_val, criterion, device)   # validate já recebe device

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} | train_acc: {train_acc:.4f} | val_acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            best_weights = copy.deepcopy(model.state_dict())

    # Salvar usando base_dir
    save_checkpoint(base_dir, dataset_name, best_weights, model_name, mode, aug, seed)
    model.load_state_dict(best_weights)

    test_metrics = evaluate_test(model, loader_test, device)
    num_params, gflops = get_model_complexity(model, device)   # <-- passando device

    # Salvar histórico e matriz usando base_dir
    save_execution(base_dir, dataset_name, model_name, mode, aug, seed, history, test_metrics)

    return {
        "seed": seed,
        "dataset": dataset_name,
        "model": model_name,
        "training_mode": mode,
        "augmentation": aug,
        "best_epoch": best_epoch,
        "val_acc_best": best_val_acc,
        "num_params": num_params,
        "gflops": gflops,
        **test_metrics
    }
    
print('Treino pronto')

In [ ]:
tempo_inicio = time.time()

results = []

for ds in datasets_config:                      
    for seed in seeds:
        for model_name in models:
            for mode in training_modes:
                for aug in augmentation_modes:
                    print(f"\n=== Dataset: {ds['name']} | Seed: {seed} | Model: {model_name} | Mode: {mode} | Aug: {aug} ===")

                    set_seed(seed)

                    # Loaders
                    transform = get_transform_train(aug)
                    dataset_train = OralCancerDataset(ds["train"], ds["folder"], transform=transform)
                    dataset_val   = OralCancerDataset(ds["val"], ds["folder"], transform=transform_validation)
                    dataset_test  = OralCancerDataset(ds["test"], ds["folder"], transform=transform_validation)

                    loader_train = DataLoader(dataset_train, batch_size=definitions["batch_size"], shuffle=True)
                    loader_val   = DataLoader(dataset_val,   batch_size=definitions["batch_size"], shuffle=False)
                    loader_test  = DataLoader(dataset_test,  batch_size=definitions["batch_size"], shuffle=False)

                    # Modelo
                    model = create_model(model_name, mode, num_classes=ds["num_classes"])
                    model = model.to(device)

                    # Loss e Optimizer (LR fixo por dataset)
                    criterion = nn.CrossEntropyLoss()
                    lr = lr_per_dataset[ds["name"]]          # <-- usando LR do dicionário
                    optimizer = optim.Adam(model.parameters(), lr=lr)

                    # Treinamento
                    result = training(
                        model, loader_train, loader_val, loader_test,
                        criterion, optimizer,
                        epochs=epochs_map[mode],            
                        model_name=model_name,
                        mode=mode,
                        aug=aug,
                        seed=seed,
                        dataset_name=ds["name"],
                        base_dir=RESULTS_DIR,              
                        device=device
                    )

                    # Adiciona caminho do checkpoint para uso posterior (Grad-CAM)
                    checkpoint_path = os.path.join(
                        RESULTS_DIR, 'checkpoints',
                        f"{ds['name']}_{model_name}_{mode}_aug{aug}_seed{seed}.pth"
                    )
                    result['model_path'] = checkpoint_path

                    results.append(result)

tempo_fim = time.time()
print(f"Tempo total de execução: {(tempo_fim - tempo_inicio)/60:.2f} minutos")

# Salvar CSV
df_results = pd.DataFrame(results)
df_results.to_csv(os.path.join(RESULTS_DIR, 'all_experiments_results.csv'), index=False)

## 10. Results Analysis: Mean & Std

Carrega o CSV e calcula média e desvio padrão para as 3 sementes.

In [ ]:
df_results = pd.read_csv(f'{RESULTS_DIR}/all_experiments_results.csv')
group_keys   = ['dataset', 'model', 'training_mode', 'augmentation']
metric_cols  = ['acc_test', 'f1_macro_test', 'f1_weighted_test']
summary = (
    df_results
    .groupby(group_keys)[metric_cols]
    .agg(['mean', 'std'])
    .round(4)
)
summary.columns = ['_'.join(c) for c in summary.columns]
summary = summary.reset_index()
summary_path = f'{RESULTS_DIR}/summary_mean_std.csv'
summary.to_csv(summary_path, index=False)
print(f'Tabela de média/desvio salva em: {summary_path}\n')
display(summary)

## 11. Final Comparative Plots

Gera gráficos comparativos de F1, matrizes de confusão (todas e melhores).

In [ ]:
df_results = pd.read_csv(f'{RESULTS_DIR}/all_experiments_results.csv')
generate_summary_plots(df_results, RESULTS_DIR)
print('Gráficos salvos em:', f'{RESULTS_DIR}/graficos/')

In [ ]:
import json, glob, re
import seaborn as sns

cm_dir  = f'{RESULTS_DIR}/confusion_matrices'
fig_dir = f'{RESULTS_DIR}/graficos/confusion_matrices'
os.makedirs(fig_dir, exist_ok=True)

history_files = glob.glob(f'{RESULTS_DIR}/histories/*.json')
print(f'\nGerando {len(history_files)} matrizes de confusão...')

class_names_map = {
    'OralEpitheliumDB': ['Healthy', 'Severe'],
    'NDB-UFES': ['Leukoplakia\nw/ Dysplasia', 'Leukoplakia\nw/o Dysplasia', 'OSCC']
}

for history_file in sorted(history_files):
    filename = os.path.basename(history_file)[:-5]
    match = re.match(
        r'^(OralEpitheliumDB|NDB-UFES)_(.+?)_(scratch|feature_extraction|fine_tuning)_(aug(True|False))_seed(\d+)$',
        filename
    )
    if not match:
        continue
    dataset_name  = match.group(1)
    model_name    = match.group(2)
    training_mode = match.group(3)
    augmentation  = match.group(5) == 'True'
    seed          = int(match.group(6))
    cm_path = f'{cm_dir}/{filename}.json'
    try:
        with open(cm_path) as f:
            cm = np.array(json.load(f))
        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        annot  = np.array([[f'{cm[i][j]}\n({cm_pct[i][j]:.1f}%)' for j in range(cm.shape[1])] for i in range(cm.shape[0])])
        labels = class_names_map.get(dataset_name, [str(i) for i in range(cm.shape[0])])
        fig, ax = plt.subplots(figsize=(10, 7))
        sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                    xticklabels=labels, yticklabels=labels)
        aug_text = 'Com Aug' if augmentation else 'Sem Aug'
        ax.set_title(f'Matriz de Confusão — {dataset_name}\n{model_name} | {training_mode} | {aug_text} | Seed: {seed}')
        ax.set_ylabel('Classe Verdadeira')
        ax.set_xlabel('Classe Predita')
        plt.tight_layout()
        plt.savefig(f'{fig_dir}/{filename}.png', dpi=100, bbox_inches='tight')
        plt.close('all')
    except FileNotFoundError:
        pass

print('\n✅ Matrizes de confusão concluídas!')

In [ ]:
# ── Matrizes dos melhores experimentos ────────────────────────────
metrics_to_check = {
    'f1_macro_test':    'F1 Macro',
    'f1_weighted_test': 'F1 Weighted',
    'acc_test':         'Acurácia',
}
criterios = {
    'geral':         ['dataset'],
    'training_mode': ['dataset', 'training_mode'],
    'modelo':        ['dataset', 'model'],
}
fig_best_dir = f'{RESULTS_DIR}/graficos/confusion_matrices_best'
os.makedirs(fig_best_dir, exist_ok=True)

for metric_col, metric_label in metrics_to_check.items():
    for criterio_name, group_cols in criterios.items():
        best_rows = df_results.loc[df_results.groupby(group_cols)[metric_col].idxmax()].drop_duplicates(subset=group_cols)
        for _, row in best_rows.iterrows():
            dataset_name  = row['dataset']
            model_name    = row['model']
            training_mode = row['training_mode']
            augmentation  = row['augmentation']
            seed          = int(row['seed'])
            filename = f'{dataset_name}_{model_name}_{training_mode}_aug{augmentation}_seed{seed}'
            cm_path  = f'{cm_dir}/{filename}.json'
            try:
                with open(cm_path) as f:
                    cm = np.array(json.load(f))
                cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
                annot  = np.array([[f'{cm[i][j]}\n({cm_pct[i][j]:.1f}%)' for j in range(cm.shape[1])] for i in range(cm.shape[0])])
                labels   = class_names_map.get(dataset_name, [str(i) for i in range(cm.shape[0])])
                aug_text = 'Com Aug' if augmentation else 'Sem Aug'
                fig, ax = plt.subplots(figsize=(10, 7))
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                            xticklabels=labels, yticklabels=labels)
                ax.set_title(f'Melhor por {criterio_name} ({metric_label})\n{dataset_name} | {model_name} | {training_mode} | {aug_text} | Seed: {seed}\n{metric_label}: {row[metric_col]:.4f}')
                ax.set_ylabel('Classe Verdadeira')
                ax.set_xlabel('Classe Predita')
                plt.tight_layout()
                fname = f'best_{criterio_name}_{metric_col}_{filename}.png'
                plt.savefig(f'{fig_best_dir}/{fname}', dpi=100, bbox_inches='tight')
                plt.show()
                plt.close('all')
            except FileNotFoundError:
                pass

print('\n✅ Matrizes dos melhores experimentos concluídas!')

## 12. Grad-CAM on Best Models

Aplica Grad‑CAM ao melhor modelo de cada dataset (maior F1 Weighted) e gera visualizações.

In [ ]:
# ── Grad-CAM nos melhores modelos ──────────────────────────────────────────
df_results = pd.read_csv(os.path.join(RESULTS_DIR, 'all_experiments_results.csv'))
ds_map = {ds['name']: ds for ds in datasets_config}

for ds_name in df_results['dataset'].unique():
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'Grad-CAM — {ds_name}')
    print(sep)

    # Melhor modelo por F1 Weighted
    best_row = df_results[df_results['dataset'] == ds_name].sort_values('f1_weighted_test', ascending=False).iloc[0]
    print(f"Melhor modelo: {best_row['model']} | {best_row['training_mode']} | aug={best_row['augmentation']} | seed={int(best_row['seed'])} | F1w={best_row['f1_weighted_test']:.4f}")

    ds_cfg = ds_map[ds_name]

    # Criar modelo
    model = create_model(best_row['model'], best_row['training_mode'], ds_cfg['num_classes'])
    # Construir caminho do checkpoint
    checkpoint_path = os.path.join(
        RESULTS_DIR, 'checkpoints',
        f"{ds_name}_{best_row['model']}_{best_row['training_mode']}_aug{best_row['augmentation']}_seed{int(best_row['seed'])}.pth"
    )
    # Carregar pesos
    state = torch.load(checkpoint_path, map_location=device)
    state = {k: v for k, v in state.items() if not k.startswith('total_')}
    model.load_state_dict(state, strict=False)
    model.to(device).eval()

    # Camada alvo
    target_layer = get_efficientnet_target_layer(model)
    print(f'Camada alvo: {target_layer.__class__.__name__}')

    # DataLoader de teste
    test_loader_gc = DataLoader(
        OralCancerDataset(ds_cfg['test'], ds_cfg['folder'], transform_validation),
        batch_size=8, shuffle=False,
        num_workers=definitions['num_workers']
    )

    print('Selecionando amostras para Grad-CAM:')
    samples = select_gradcam_samples(model, test_loader_gc, device, n_per_class=2)
    print(f'{len(samples)} amostras selecionadas no total')

    gc_dir = os.path.join(RESULTS_DIR, 'gradcam', ds_name)
    os.makedirs(gc_dir, exist_ok=True)

    for i, sample in enumerate(samples):
        status = 'CORRETA' if sample['correct'] else 'INCORRETA'
        cls_name = ds_cfg['class_names'][sample['label']]
        pred_name = ds_cfg['class_names'][sample['pred']]
        print(f'  [{i+1}/{len(samples)}] Classe: {cls_name} | Predição: {pred_name} | {status}')
        visualize_gradcam(model, target_layer, sample, ds_cfg['class_names'], i, gc_dir)

    print(f'Visualizações salvas em: {gc_dir}')

print('\n✅ Grad-CAM concluído!')